In [2]:
from Bio import Entrez, SeqIO
import requests, json, os, time

Entrez.email = 'diyamsn7@gmail.com'   # NCBI requires a valid email
Entrez.api_key = 'f7a1d5c26f72cf925b55c671ba5f294df508'               # Optional: get a free key at ncbi.nlm.nih.gov/account

In [7]:
handle = Entrez.efetch(db='protein', id='NP_777596', rettype='fasta', retmode='text')
pcsk9_seq = handle.read()
handle.close()

with open('../data/raw/pcsk9_protein.fasta', 'w') as f:
    f.write(pcsk9_seq)

print('PCSK9 sequence saved.')

PCSK9 sequence saved.


In [2]:
import requests
import time

def download_clinvar_gene(gene_symbol, filename):
    # Use the ClinVar esearch then efetch with explicit gene filter
    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'
    
    # Search
    search_url = f'{base}esearch.fcgi?db=clinvar&term={gene_symbol}[gene]+AND+pathogenic[clinsig]&retmax=500&retmode=json'
    r = requests.get(search_url)
    ids = r.json()['esearchresult']['idlist']
    print(f'{gene_symbol}: {len(ids)} IDs found')
    time.sleep(1)
    
    # Fetch
    ids_str = ','.join(ids)
    fetch_url = f'{base}efetch.fcgi?db=clinvar&id={ids_str}&rettype=vcv&retmode=xml&is_variationid'
    r = requests.get(fetch_url)
    
    with open(filename, 'wb') as f:
        f.write(r.content)
    print(f'Saved {filename}')
    time.sleep(2)

download_clinvar_gene('PCSK9', '../data/raw/clinvar_pcsk9.xml')
download_clinvar_gene('LDLR', '../data/raw/clinvar_ldlr.xml')
download_clinvar_gene('APOB', '../data/raw/clinvar_apob.xml')

PCSK9: 500 IDs found
Saved ../data/raw/clinvar_pcsk9.xml
LDLR: 500 IDs found
Saved ../data/raw/clinvar_ldlr.xml
APOB: 500 IDs found
Saved ../data/raw/clinvar_apob.xml


In [12]:
import requests, json

# Use the target endpoint instead — more reliable for gene-based assay lookup
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/target/genesymbol/PCSK9/aids/JSON'
response = requests.get(url)
data = response.json()

# Print the raw response so we can see the actual structure
print(json.dumps(data, indent=2)[:500])

{
  "IdentifierList": {
    "AID": [
      1904,
      208829,
      208830,
      208831,
      208832,
      208833,
      208834,
      208835,
      208836,
      624099,
      651810,
      651811,
      743454,
      1053208,
      1117281,
      1117357,
      1159506,
      1159578,
      1159584,
      1224826,
      1224828,
      1224830,
      1345955,
      1390359,
      1390360,
      1390361,
      1390362,
      1390363,
      1390364,
      1390365,
      1390366,
      1390367


In [17]:
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/target/genesymbol/PCSK9/aids/JSON'
response = requests.get(url)
aids = response.json()['IdentifierList']['AID']
print(f'Found {len(aids)} PCSK9-related bioassays in PubChem')

with open('../data/raw/pubchem_pcsk9_assay_ids.json', 'w') as f:
    json.dump(aids, f)

Found 142 PCSK9-related bioassays in PubChem


In [16]:
assay_data = []
for aid in aids[:10]:
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/JSON'
    r = requests.get(url)
    if r.status_code == 200:
        assay_data.append(r.json())
    time.sleep(0.5)

with open('../data/raw/pubchem_assay_details.json', 'w') as f:
    json.dump(assay_data, f)
print('Bioassay data saved.')

Bioassay data saved.
